# Part 10
## 10.1 & 10.2

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_json('/media/trong-thanh/Data/University/Fourth Year/NLP/experiments/ex1/data.json', lines=True)
texts = df['text'].fillna('').tolist()
print(f'So luong documents: {len(texts):,}')


So luong documents: 30,000


In [2]:
vectorizer = CountVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b\w+\b')
X = vectorizer.fit_transform(texts)

tfidf_trans = TfidfTransformer()
X_tfidf = tfidf_trans.fit_transform(X)
print(f'X_tfidf shape: {X_tfidf.shape}')


X_tfidf shape: (30000, 193521)


In [3]:
def search(query, k=5):
    q_vec = tfidf_trans.transform(vectorizer.transform([query]))
    sims = cosine_similarity(q_vec, X_tfidf).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    
    rows = []
    for rank, idx in enumerate(top_idx, 1):
        rows.append({
            'Rank': rank,
            'Doc ID': idx,
            'Similarity': round(float(sims[idx]), 4),
            'Preview': texts[idx].replace('\n', ' ')[:100] + '...'
        })
    return pd.DataFrame(rows)


## 10.3 & 10.4

In [4]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in queries:
    print(f"Query: {q}")
    print(search(q).to_string(index=False))
    print()


Query: medical image classification
 Rank  Doc ID  Similarity                                                                                                 Preview
    1   18971      0.4237 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commission...
    2    8527      0.3537 History of maize classification. How races used in classification. Geographical distribution. Existi...
    3   19908      0.2609 Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-scree...
    4   12658      0.2490 This guidance is for pharmacists who handle, use and sell/supply medical devices. What to do with me...
    5   15682      0.2486 - Group Image: Provided functionality of group's image, user can now view and change the group's ima...

Query: transformer language model
 Rank  Doc ID  Similarity                                                                                                 Preview
    1   27936      

### Nhận xét
- Tốc độ tìm kiếm nhanh nhờ nhân vô hướng ma trận thưa.
- Các tài liệu đứng đầu đều chứa đúng các từ khóa truy vấn.
- Hạn chế: Không bắt được từ đồng nghĩa nếu văn bản không chứa đúng từ vựng của query.